# Week 45, Day 1: Tree vocabulary

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w045d1.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
class Node:
    def __init__(self, value, children=()):
        self.value = value
        self.children = list(children)

def show(node, depth=0):
    print("  " * depth + node.value)
    for child in node.children:
        show(child, depth + 1)

tree = Node("home", [
    Node("ada", [Node("cv.pdf"), Node("photos", [Node("cat.jpg")])]),
    Node("bo", [Node("notes.txt")]),
])
show(tree)

In [ ]:
class Node:
    def __init__(self, value, children=()):
        self.value = value
        self.children = list(children)

def from_parents(parent_of):
    """Build linked nodes from a dict child -> parent (the root maps to None)."""
    nodes = {name: Node(name) for name in parent_of}
    root = None
    for name, parent in parent_of.items():
        if parent is None:
            root = nodes[name]
        else:
            nodes[parent].children.append(nodes[name])
    return root

def to_nested(node):
    return (node.value, [to_nested(c) for c in node.children])

staff = {"Ngozi": None, "Ade": "Ngozi", "Bea": "Ngozi", "Chi": "Ade", "Dan": "Ade", "Eve": "Bea"}
boss = from_parents(staff)
print(to_nested(boss))

In [ ]:
class Node:
    def __init__(self, value, children=()):
        self.value = value
        self.children = list(children)

def size(node):
    return 1 + sum(size(c) for c in node.children)

def height(node):
    return 1 + max((height(c) for c in node.children), default=-1)     # a leaf: 1 + -1 = 0

def leaves(node):
    if not node.children:
        return [node.value]
    return [leaf for c in node.children for leaf in leaves(c)]

def count_at_depth(node, depth):
    if depth == 0:
        return 1
    return sum(count_at_depth(c, depth - 1) for c in node.children)

tree = Node("home", [
    Node("ada", [Node("cv.pdf"), Node("photos", [Node("cat.jpg")])]),
    Node("bo", [Node("notes.txt")]),
])
print(size(tree), height(tree), leaves(tree), [count_at_depth(tree, d) for d in range(4)])

In [ ]:
class Node:
    def __init__(self, value, children=()):
        self.value = value
        self.children = list(children)

def path_to(node, target):
    """The values from node down to target, or None if target isn't in this subtree."""
    if node.value == target:
        return [node.value]
    for child in node.children:
        rest = path_to(child, target)
        if rest is not None:
            return [node.value] + rest
    return None

tree = Node("home", [Node("ada", [Node("cv.pdf"), Node("photos", [Node("cat.jpg")])]), Node("bo")])
print(path_to(tree, "cat.jpg"), path_to(tree, "dog.jpg"))

In [ ]:
staff = {"Ngozi": None, "Ade": "Ngozi", "Bea": "Ngozi", "Chi": "Ade", "Dan": "Ade", "Eve": "Bea"}

def ancestors(parent_of, name):
    """name, its parent, its grandparent, ..., up to the root."""
    out = []
    while name is not None:
        out.append(name)
        name = parent_of[name]
    return out

def lowest_common_ancestor(parent_of, a, b):
    above_a = set(ancestors(parent_of, a))
    for name in ancestors(parent_of, b):
        if name in above_a:
            return name

print(lowest_common_ancestor(staff, "Chi", "Dan"), lowest_common_ancestor(staff, "Chi", "Eve"), lowest_common_ancestor(staff, "Chi", "Ade"))

> **Key idea.** A tree has one root; every other node has exactly one parent, and nodes without children are leaves. Depth is edges down from the root; height is edges on the longest path down to a leaf (a leaf is 0; this course gives an empty tree −1). Store trees as linked nodes with children (walking down), as a parent map (walking up), or as nested data (writing them down). Size, height, leaves and paths are short recursive functions that combine answers from the children, O(n) in all; recursion depth equals the tree's height. A tree with n nodes has n − 1 edges and no cycles. The lowest common ancestor of two nodes is the first of one's ancestors that's also an ancestor of the other.

### Exercise: Size, leaves and degree

`Node` is given. Write `size(node)` (the number of nodes), `leaf_count(node)`, and `max_degree(node)` (the most children any node in the tree has), each recursively.

In [ ]:
class Node:
    def __init__(self, value, children=()):
        self.value = value
        self.children = list(children)

def size(node):
    ___

def leaf_count(node):
    ___

def max_degree(node):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_basics():\n    \"counting\"\n    t = Node(\"a\", [Node(\"b\", [Node(\"d\"), Node(\"e\"), Node(\"f\")]), Node(\"c\", [Node(\"g\")])])\n    assert size(t) == 7 and leaf_count(t) == 4 and max_degree(t) == 3\n    one = Node(\"x\")\n    assert size(one) == 1 and leaf_count(one) == 1 and max_degree(one) == 0")

### Exercise: Height and depth

Write `height(node)` (edges on the longest path down; a leaf is 0) and `depth_of(node, target)`: the depth of the first node whose value is `target` (searching children in order), or `None` if there's none.

In [ ]:
class Node:
    def __init__(self, value, children=()):
        self.value = value
        self.children = list(children)

def height(node):
    ___

def depth_of(node, target):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_height():\n    \"height and depth\"\n    t = Node(\"a\", [Node(\"b\", [Node(\"d\", [Node(\"h\")])]), Node(\"c\")])\n    assert height(t) == 3 and height(Node(\"x\")) == 0 and height(t.children[1]) == 0\n    assert depth_of(t, \"a\") == 0 and depth_of(t, \"h\") == 3 and depth_of(t, \"c\") == 1 and depth_of(t, \"z\") is None")

### Exercise: From a parent map

Write `build(parent_of)`: given a dictionary from each name to its parent's name (`None` for the root), return the root `Node`, with each node's children in the order they appear in the dictionary. Raise `ValueError` if there isn't exactly one root, or if a parent named isn't in the dictionary. Then write `to_parents(root)`, which turns a tree back into such a dictionary (in depth-first order: a node, then its children's subtrees in order).

In [ ]:
class Node:
    def __init__(self, value, children=()):
        self.value = value
        self.children = list(children)

def build(parent_of):
    ___

def to_parents(root):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def nested(n):\n    return (n.value, [nested(c) for c in n.children])\ndef test_build():\n    \"building\"\n    staff = {\"Ngozi\": None, \"Ade\": \"Ngozi\", \"Bea\": \"Ngozi\", \"Chi\": \"Ade\", \"Dan\": \"Ade\", \"Eve\": \"Bea\"}\n    root = build(staff)\n    assert nested(root) == (\"Ngozi\", [(\"Ade\", [(\"Chi\", []), (\"Dan\", [])]), (\"Bea\", [(\"Eve\", [])])])\n    assert to_parents(root) == {\"Ngozi\": None, \"Ade\": \"Ngozi\", \"Chi\": \"Ade\", \"Dan\": \"Ade\", \"Bea\": \"Ngozi\", \"Eve\": \"Bea\"}\n    for bad in [{\"a\": None, \"b\": None}, {\"a\": \"b\", \"b\": \"a\"}, {\"a\": None, \"b\": \"zed\"}]:\n        try:\n            build(bad)\n            assert False, bad\n        except ValueError:\n            pass")

### Exercise: An outline

Write `outline(node)`: a list of lines showing the tree as a numbered outline, like a document's headings. The root is just its value; its children are `1 Value`, `2 Value`, ...; their children `1.1 Value`, `1.2 Value`; and so on, each line indented by two spaces per level below the root.

In [ ]:
class Node:
    def __init__(self, value, children=()):
        self.value = value
        self.children = list(children)

def outline(node):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_outline():\n    \"an outline\"\n    doc = Node(\"Guide\", [Node(\"Start\", [Node(\"Install\"), Node(\"Run\")]), Node(\"Use\", [Node(\"Basics\", [Node(\"Keys\")])]), Node(\"Help\")])\n    assert outline(doc) == [\"Guide\", \"  1 Start\", \"    1.1 Install\", \"    1.2 Run\", \"  2 Use\", \"    2.1 Basics\", \"      2.1.1 Keys\", \"  3 Help\"]\n    assert outline(Node(\"x\")) == [\"x\"]")

### Exercise: Common ancestors

Write `common_ancestor(parent_of, a, b)` with a parent map (a node counts as its own ancestor), and `distance(parent_of, a, b)`: the number of edges on the path between `a` and `b`, going up to their common ancestor and down again. Unknown names raise `KeyError`.

In [ ]:
def common_ancestor(parent_of, a, b):
    ___

def distance(parent_of, a, b):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_lca():\n    \"ancestors\"\n    t = {\"r\": None, \"a\": \"r\", \"b\": \"r\", \"c\": \"a\", \"d\": \"a\", \"e\": \"c\", \"f\": \"b\"}\n    assert common_ancestor(t, \"e\", \"d\") == \"a\" and common_ancestor(t, \"e\", \"f\") == \"r\" and common_ancestor(t, \"c\", \"e\") == \"c\" and common_ancestor(t, \"r\", \"r\") == \"r\"\n    assert distance(t, \"e\", \"d\") == 3 and distance(t, \"e\", \"f\") == 5 and distance(t, \"c\", \"e\") == 1 and distance(t, \"b\", \"b\") == 0\n    try:\n        common_ancestor(t, \"e\", \"zz\")\n        assert False\n    except KeyError:\n        pass")

### Exercise: Every root-to-leaf path

Write `all_paths(node)`: every path from the root to a leaf, as `"a/b/c"` strings, in left-to-right order.

In [ ]:
class Node:
    def __init__(self, value, children=()):
        self.value = value
        self.children = list(children)

def all_paths(node):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_paths():\n    \"paths\"\n    t = Node(\"home\", [Node(\"ada\", [Node(\"cv.pdf\"), Node(\"photos\", [Node(\"cat.jpg\")])]), Node(\"bo\")])\n    assert all_paths(t) == [\"home/ada/cv.pdf\", \"home/ada/photos/cat.jpg\", \"home/bo\"] and all_paths(Node(\"x\")) == [\"x\"]")

### Bug hunt: One shared family

`Node`'s default `children=[]` is a single list shared by every node made without children, so adding a child to one leaf adds it to all of them: this prints `1 1`. Give each node its own list, so it prints `1 0`.

In [ ]:
class Node:
    def __init__(self, value, children=[]):
        self.value = value
        self.children = children

a, b = Node("a"), Node("b")
a.children.append(Node("c"))
print(len(a.children), len(b.children))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"a list each\"\n    expect_output(\"1 0\")")

### Bug hunt: Counting nodes, not edges

This course measures height in **edges**, so a single node has height 0. This function counts **nodes** instead, so it's one too big everywhere: it prints `1 3`. Fix it so it prints `0 2`.

In [ ]:
class Node:
    def __init__(self, value, children=()):
        self.value = value
        self.children = list(children)

def height(node):
    return 1 + max((height(c) for c in node.children), default=0)

print(height(Node("x")), height(Node("a", [Node("b", [Node("c")])])))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"edges\"\n    expect_output(\"0 2\")")

### Bug hunt: Found, then forgotten

`path_to` finds the target in a child's subtree but doesn't **return** what it found, so the answer is lost on the way back up and it prints `None`. Fix it so it prints `['a', 'b', 'd']`.

In [ ]:
class Node:
    def __init__(self, value, children=()):
        self.value = value
        self.children = list(children)

def path_to(node, target):
    if node.value == target:
        return [node.value]
    for child in node.children:
        rest = path_to(child, target)
        if rest is not None:
            [node.value] + rest
    return None

t = Node("a", [Node("b", [Node("c"), Node("d")]), Node("e")])
print(path_to(t, "d"))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"return it\"\n    expect_output(\"['a', 'b', 'd']\")")

### Puzzle: Counting leaves

Arrange the lines into a recursive function counting the leaves of a tree. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
# For the tree   r
#               / \
#              a   b
#             / \   \
#            c   d   e
#                    |
#                    f

p = ___   # the height of the tree (in edges)
q = ___   # the depth of f
r = ___   # the number of leaves
s = ___   # the number of edges
t = ___   # the lowest common ancestor of c and f

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == 3 and q == 3 and r == 3, \"p, q and r\"\n    assert s == 6 and t == \"r\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: folder sizes

A disc listing gives file paths and sizes, like `("docs/cv.pdf", 120)`. Write `FolderTree(listing)`, which builds a tree of folders and files (the root is a folder named `""`); then:

- `size(path="")`: the total size of everything under a folder (or a file's own size), where `path` is like `"docs/old"`; a missing path raises `KeyError`;
- `largest_folders(k)`: the `k` largest **folders** (not counting the root), as `(path, size)` pairs, largest first, then by path;
- `report(max_depth)`: lines like `"docs 300"`, one per folder, depth-first with children in alphabetical order, indented two spaces per level, showing folders down to `max_depth` levels below the root (folders only, not files);
- `deletable(target)`: the smallest folder whose size is **at least** `target`, as a path (`None` if there's none).

A path that appears as both a file and a folder (`"a"` as a file and `"a/b"`) raises `ValueError`.

In [ ]:
class FolderTree:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("LISTING = [(\"docs/cv.pdf\", 120), (\"docs/old/letter.txt\", 30), (\"docs/old/photo.png\", 150), (\"music/song.mp3\", 400),\n           (\"readme.txt\", 5), (\"docs/notes.txt\", 20), (\"music/live/gig.mp3\", 900)]\ndef test_sizes():\n    \"sizes\"\n    t = FolderTree(LISTING)\n    assert t.size() == 1625 and t.size(\"docs\") == 320 and t.size(\"docs/old\") == 180 and t.size(\"readme.txt\") == 5 and t.size(\"music/live\") == 900\n    try:\n        t.size(\"nowhere\")\n        assert False\n    except KeyError:\n        pass\ndef test_largest():\n    \"largest folders\"\n    t = FolderTree(LISTING)\n    assert t.largest_folders(2) == [(\"music\", 1300), (\"music/live\", 900)] and t.largest_folders(10)[-1] == (\"docs/old\", 180)\n    assert t.deletable(200) == \"docs\" and t.deletable(1000) == \"music\" and t.deletable(5000) is None and t.deletable(100) == \"docs/old\"\ndef test_report():\n    \"the report\"\n    t = FolderTree(LISTING)\n    assert t.report(1) == [\"docs 320\", \"music 1300\"]\n    assert t.report(2) == [\"docs 320\", \"  old 180\", \"music 1300\", \"  live 900\"]\ndef test_clash():\n    \"file and folder\"\n    try:\n        FolderTree([(\"a\", 1), (\"a/b\", 2)])\n        assert False\n    except ValueError:\n        pass\n    try:\n        FolderTree([(\"a/b\", 2), (\"a\", 1)])\n        assert False\n    except ValueError:\n        pass")

## Warm-up

A folder structure is a tree: each node has a value and a list of **children**, and a few lines of recursion print the whole thing, however deep it goes. That pattern (do something with this node, then the same for each child) is the heart of today.

## The words

```text
                home                 <- the root: the one node with no parent       depth 0
              /      \
           ada        bo             <- children of home; siblings of each other     depth 1
          /    \        \
     cv.pdf   photos   notes.txt     <- cv.pdf and notes.txt are leaves              depth 2
                 |
              cat.jpg                <- a leaf, deepest node                         depth 3

     the height of the tree is 3: the longest path from the root down to a leaf, in edges
```

| Word | Meaning |
|---|---|
| **node** | one item in the tree |
| **edge** | the link between a parent and a child |
| **root** | the single node at the top, with no parent |
| **parent**, **child** | a node directly above, or directly below, another |
| **siblings** | children of the same parent |
| **leaf** | a node with no children; any other node is **internal** |
| **ancestor**, **descendant** | a node anywhere above (parent, grandparent, ...), or anywhere below |
| **subtree** | a node together with all its descendants: itself a tree |
| **depth** of a node | the number of edges from the root down to it (the root's depth is 0) |
| **level** | all the nodes at the same depth |
| **height** of a node | the number of edges on the longest path from it down to a leaf (a leaf's height is 0) |
| **height** of a tree | the height of its root; this course says an empty tree has height −1 |
| **degree** of a node | how many children it has |

Two conventions vary between books: some count height and depth in **nodes** rather than edges (so a single node has height 1), and some put the root at level 1. This course counts **edges** and starts at 0, but always check which a problem means.

## Three ways to store a tree

**Linked nodes** (as above): each node holds its value and a list of its children. Good for walking **down** from the root, which most algorithms do.

**A parent map**: a dictionary from each node to its parent (the root's parent is `None`). Good for walking **up**, finding ancestors, and loading trees from tables or files, where each row says "this item belongs to that one" (employees and managers, comments and replies).

**Nested data**: tuples, lists or dictionaries inside each other, like Week 10's nested dictionaries and JSON (Week 23). Good for writing trees down and reading them in.

Notice the children come out in the order the dictionary listed them: trees often keep their children in a meaningful order (the order of headings in a document), and the representation should preserve it.

## Small recursive tree functions

Because every subtree is itself a tree, most questions about a tree have a two-line recursive answer: answer it for each child's subtree, then combine. Week 15's rules apply: the base case is usually a leaf (or an empty tree), and each call works on a smaller subtree.

`max(..., default=-1)` handles leaves neatly: with no children, the `max` is -1, and the leaf's height is 0. Each function visits every node once: O(n) for a tree of n nodes.

A caution from Week 15: recursion goes as deep as the tree is tall. A tree shaped like a long chain (each node with one child) a few thousand nodes tall would hit Python's recursion limit. Most real trees are short and wide (a folder tree is rarely more than 20 deep), but when one might not be, use an explicit stack (Week 43).

## Paths and common ancestors

The **path** from the root to a node is found by searching down: a node is on the path if the target is in its subtree.

With a **parent map**, going up is easier still. The **lowest common ancestor** of two nodes (the deepest node that's an ancestor of both, such as the manager two employees share) is found by collecting one node's ancestors in a set, then walking up from the other until you meet one:

Each of these is O(h), where h is the tree's height: fast for short trees, however many nodes they have. (A node counts as its own ancestor here, so the common ancestor of Chi and Ade is Ade.)

## What makes it a tree

A tree is a set of nodes and edges with three properties, any two of which imply the third:

- it's **connected**: every node can be reached from the root;
- it has **no cycles**: you can't follow edges round in a loop (Week 42's linked-list cycles would break every tree algorithm);
- it has exactly **n − 1 edges** for n nodes: every node except the root has exactly one edge up to its parent.

If a node could have two parents, or the links could form a loop, it would be a **graph** instead, which is coming later in this phase. A parent map can only describe a tree if following parents from any node always reaches the root without repeating, which is worth checking when the data comes from outside (Week 37).

## Practice

## Quiz

## Daily challenge

## Before you go

Draw the folder tree on your own computer's home folder, three levels deep, and label its root, a leaf, a node's depth and the tree's height. Tomorrow: binary trees, where every node has at most two children, called left and right.